In [ ]:
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy

## Training the MountainCar-v0 agent

In [ ]:
env = gym.make('MountainCar-v0', render_mode="human")

agent = DQN('MlpPolicy', env, learning_rate=1e-3)

agent.learn(total_timesteps=25000)

### Evaluate MountainCar-v0 agent

In [ ]:
mean_reward, std_reward = evaluate_policy(agent, agent.get_env(), n_eval_episodes=10)
print(f"Mean reward: {mean_reward}, Std reward: {std_reward}")

## Saving the agent

In [ ]:
agent.save("../agents/DQN_mountain_car_v0_agent")

## Viewing the agent

In [ ]:
state, info = env.reset()

for t in range(5000):
    action, _states = agent.predict(state)
    next_state, reward, done, truncated, info = env.step(action)
    state = next_state
    env.render()

    if done or truncated:
        break

env.close()

## Vectorized Envs

In [22]:
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3 import PPO

def make_env(env_name, rank, seed=0):

    def _init():
        env = gym.make(env_name)
        env.reset(seed=seed + rank)
        return env

    set_random_seed(seed)
    return _init

        

In [23]:
env_name = "Pendulum-v1"
num_processes = 2
seed = 43

env = SubprocVecEnv(
    [
        make_env(env_name, i, seed=seed)
        for i in range(num_processes)
    ]
)

In [24]:
model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    seed=seed
)

Using cuda device


/home/amosnyirenda/miniconda3/envs/rl/lib/python3.12/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


In [25]:
model.learn(total_timesteps=100_000)

-----------------------------
| time/              |      |
|    fps             | 1534 |
|    iterations      | 1    |
|    time_elapsed    | 2    |
|    total_timesteps | 4096 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 1079        |
|    iterations           | 2           |
|    time_elapsed         | 7           |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.002747346 |
|    clip_fraction        | 0.0132      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.42       |
|    explained_variance   | -0.00443    |
|    learning_rate        | 0.0003      |
|    loss                 | 3.57e+03    |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.00145    |
|    std                  | 0.997       |
|    value_loss           | 8.76e+03    |
----------------------------------

In [26]:
model.save("../agents/PP0_pendulum_v1")

env.close()

In [27]:
eval_env = gym.make(
    env_name,
    render_mode="human"
)

state, info = eval_env.reset(seed=seed)

In [28]:
for _ in range(1000):

    action, _states = model.predict(
        state,
        deterministic=True
    )

    next_state, reward, terminated, truncated, info = eval_env.step(action)

    state = next_state

    if terminated or truncated:
        state, info = eval_env.reset()

In [29]:
eval_env.close()